In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings("ignore")

In [48]:
df = pd.read_csv("./Data/cleaned_data.csv")
df.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_F,Sex_M,ChestPainType_ASY,...,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ExerciseAngina_N,ExerciseAngina_Y,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0,0,0,1,0,...,0,0,0,1,0,1,0,0,0,1
1,49,160,180,0,156,1,1,1,0,0,...,1,0,0,1,0,1,0,0,1,0
2,37,130,283,0,98,0,0,0,1,0,...,0,0,0,0,1,1,0,0,0,1
3,48,138,214,0,108,1,1,1,0,1,...,0,0,0,1,0,0,1,0,1,0
4,54,150,195,0,122,0,0,0,1,0,...,1,0,0,1,0,1,0,0,0,1


In [49]:
X = df.drop(columns="HeartDisease",axis=1)
Y = df["HeartDisease"].round().astype(int)

X.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_F,Sex_M,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ExerciseAngina_N,ExerciseAngina_Y,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0,0,1,0,1,0,0,0,1,0,1,0,0,0,1
1,49,160,180,0,156,1,1,0,0,0,1,0,0,1,0,1,0,0,1,0
2,37,130,283,0,98,0,0,1,0,1,0,0,0,0,1,1,0,0,0,1
3,48,138,214,0,108,1,1,0,1,0,0,0,0,1,0,0,1,0,1,0
4,54,150,195,0,122,0,0,1,0,0,1,0,0,1,0,1,0,0,0,1


In [50]:
Y.head()

0    0
1    1
2    0
3    1
4    0
Name: HeartDisease, dtype: int64

In [51]:
num_featues = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(exclude="int").columns

# print(num_featues, cat_features)

In [52]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer([
    ("OneHotEncoder", oh_transformer, cat_features),
    ("StandardScalar", numeric_transformer, num_featues),
])

In [53]:
X = preprocessor.fit_transform(X)

In [54]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((734, 20), (184, 20))

In [55]:

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, r2_score

models = {
    "Logistic Regression": LogisticRegression(),
    "KNN Classifier": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "AdaBoost Classifier": AdaBoostClassifier()
}

models_list = []
accuracy_list = []

for name, model in models.items():
    model.fit(X_train, Y_train)

    Y_train_pred = model.predict(X_train)
    Y_test_pred = model.predict(X_test)

    print(name)
    models_list.append(name)

    print("Training Performance")
    print("Accuracy:", accuracy_score(Y_train, Y_train_pred))
    print("F1 Score:", f1_score(Y_train, Y_train_pred))
    
    print("Test Performance")
    print("Accuracy:", accuracy_score(Y_test, Y_test_pred))
    print("F1 Score:", f1_score(Y_test, Y_test_pred))
    
    accuracy_list.append(accuracy_score(Y_test, Y_test_pred))

    print("=" * 40)
    print("\n")


Logistic Regression
Training Performance
Accuracy: 0.8664850136239782
F1 Score: 0.8796068796068796
Test Performance
Accuracy: 0.8641304347826086
F1 Score: 0.8792270531400966


KNN Classifier
Training Performance
Accuracy: 0.8773841961852861
F1 Score: 0.8902439024390244
Test Performance
Accuracy: 0.8369565217391305
F1 Score: 0.8543689320388349


Decision Tree
Training Performance
Accuracy: 1.0
F1 Score: 1.0
Test Performance
Accuracy: 0.8369565217391305
F1 Score: 0.84375


Random Forest
Training Performance
Accuracy: 1.0
F1 Score: 1.0
Test Performance
Accuracy: 0.8695652173913043
F1 Score: 0.8857142857142857


AdaBoost Classifier
Training Performance
Accuracy: 0.8746594005449592
F1 Score: 0.8858560794044665
Test Performance
Accuracy: 0.8586956521739131
F1 Score: 0.8725490196078431




In [56]:
pd.DataFrame(list(zip(models_list, accuracy_list)), columns=["Model Name", "R2_score"]).sort_values(by=["R2_score"],ascending=False)

,Model Name,R2_score
3,Random Forest,0.869565
0,Logistic Regression,0.864130
4,AdaBoost Classifier,0.858696
2,Decision Tree,0.836957
1,KNN Classifier,0.836957


In [58]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Create the base model
rf = RandomForestClassifier(random_state=42)

# Instantiate the grid search model
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# Fit the grid search to the data
grid_search.fit(X_train, Y_train)

# Best parameters and score
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy: %.2f%%" % (grid_search.best_score_ * 100))


Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best Parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 300}
Best Accuracy: 86.64%


In [59]:
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)
score = accuracy_score(Y_test, y_pred) * 100
print("Tuned Model Accuracy: %.2f%%" % score)

Tuned Model Accuracy: 86.41%
